# Notebook 08: sklearn Pipelines

## Why This Matters

Pipelines are the single most important sklearn construct for production ML. They prevent data leakage in cross-validation, make the preprocessing + model a single deployable object, enable hyperparameter search over preprocessing choices, and make code cleaner. Mastering Pipeline and ColumnTransformer is the difference between notebook code and production code.

In [ ]:
%pip install -q scikit-learn pandas numpy matplotlib

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV, train_test_split
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import roc_auc_score
from sklearn.datasets import make_classification
import warnings; warnings.filterwarnings("ignore")
np.random.seed(42)
print("Ready.")

## 1. Pipeline: Chain Transformers + Estimator

`Pipeline([('name', step), ...])` chains transformers with a final estimator.

**Key properties**:
- `fit()` calls `fit_transform()` on each transformer, then `fit()` on the estimator
- `predict()` calls `transform()` on each transformer, then `predict()` on the estimator
- With `cross_val_score`, each fold refits all transformers on training data only
- `set_params(step__param=value)` lets you set hyperparameters on any step

**Why pipelines prevent leakage**: sklearn calls `pipe.fit(X_train_fold, y_train_fold)` for each fold. Transformers are always fit on training fold only.

In [ ]:
X, y = make_classification(n_samples=1000, n_features=20, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

pipe = Pipeline([("imputer", SimpleImputer()), ("scaler", StandardScaler()), ("clf", LogisticRegression(C=1.0, max_iter=500))])
pipe.fit(X_tr, y_tr)
print(f"Test AUC: {roc_auc_score(y_te, pipe.predict_proba(X_te)[:,1]):.4f}")

param_grid = {"clf__C": [0.01, 0.1, 1.0, 10.0], "scaler": [StandardScaler(), None]}
gs = GridSearchCV(pipe, param_grid, cv=5, scoring="roc_auc", n_jobs=-1)
gs.fit(X_tr, y_tr)
print(f"Best CV AUC: {gs.best_score_:.4f}, Best params: {gs.best_params_}")
cv = cross_val_score(pipe, X, y, cv=5)
print(f"5-fold CV: {cv.mean():.4f} +/- {cv.std():.4f}")
print("Pipeline ensures scaler is re-fit on each training fold only - no leakage.")

## 2. ColumnTransformer: Different Transforms for Different Columns

`ColumnTransformer` applies different transformations to different column subsets, then concatenates results.

**Typical pattern for mixed-type tabular data**:
- Numeric: impute with median -> scale
- Low-cardinality categorical: impute with mode -> one-hot encode
- High-cardinality categorical: impute with mode -> ordinal encode (or target encode)

In [ ]:
n = 2000
df = pd.DataFrame({"age": np.random.randint(18, 70, n).astype(float), "income": np.random.lognormal(10, 0.8, n), "hours": np.random.normal(40, 10, n), "education": np.random.choice(["hs","college","grad"], n), "marital": np.random.choice(["single","married","divorced"], n), "occupation": np.random.choice([f"job_{i}" for i in range(20)], n)})
for col in ["age","income"]: df.loc[np.random.choice(n, 200), col] = np.nan
target = ((df.income.fillna(50000) > 60000) & (df.hours > 38)).astype(int)

num_t = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())])
cat_t = Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
prep = ColumnTransformer([("num", num_t, ["age","income","hours"]), ("cat", cat_t, ["education","marital"])], remainder="drop")
full = Pipeline([("prep", prep), ("clf", GradientBoostingClassifier(n_estimators=100, random_state=42))])

X_tr, X_te, y_tr, y_te = train_test_split(df, target, test_size=0.2, stratify=target, random_state=42)
cv = cross_val_score(full, X_tr, y_tr, cv=5, scoring="roc_auc")
print(f"Full Pipeline CV AUC: {cv.mean():.4f} +/- {cv.std():.4f}")
full.fit(X_tr, y_tr)
print(f"Test AUC: {roc_auc_score(y_te, full.predict_proba(X_te)[:,1]):.4f}")
print(f"Transformed shape: {prep.fit_transform(X_tr).shape}")

## 3. Custom Transformers in Pipelines

Inherit `BaseEstimator + TransformerMixin`. Requirements:
1. Set all parameters in `__init__`
2. All learning in `fit()`
3. `transform()` uses learned state from `fit()`
4. `TransformerMixin` provides `fit_transform()` for free

In [ ]:
class PercentileClipper(BaseEstimator, TransformerMixin):
    def __init__(self, low=1, high=99):
        self.low = low; self.high = high
    
    def fit(self, X, y=None):
        X = np.array(X)
        self.lower_ = np.percentile(X, self.low, axis=0)
        self.upper_ = np.percentile(X, self.high, axis=0)
        return self
    
    def transform(self, X):
        return np.clip(np.array(X, dtype=float), self.lower_, self.upper_)

X_demo = np.random.randn(100, 3); X_demo[0, 0] = 100
clipper = PercentileClipper(low=5, high=95)
X_clipped = clipper.fit_transform(X_demo)
print(f"Before clipping: max f0 = {X_demo[:,0].max():.2f}")
print(f"After clipping:  max f0 = {X_clipped[:,0].max():.2f}")

X2, y2 = make_classification(n_samples=500, n_features=10, random_state=42)
clip_pipe = Pipeline([("clip", PercentileClipper()), ("sc", StandardScaler()), ("clf", LogisticRegression(max_iter=500))])
cv2 = cross_val_score(clip_pipe, X2, y2, cv=5)
print(f"\nCustom clipper pipeline CV: {cv2.mean():.4f} +/- {cv2.std():.4f}")
print("Custom transformer works seamlessly with GridSearchCV and cross_val_score.")

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| Pipeline | Chains transformers + estimator; prevents leakage in cross-validation |
| ColumnTransformer | Different transforms per column type; concatenates results |
| step__param syntax | Access nested params in GridSearchCV |
| Custom transformer | BaseEstimator + TransformerMixin; learn in fit(), apply in transform() |
| remainder='passthrough' | Keep non-specified columns in ColumnTransformer |

### Common Pitfalls
- Not using Pipeline -> manual preprocessing loops that are fragile and leak
- Fitting ColumnTransformer before train/test split
- Setting mutable defaults in `__init__` instead of simple values
- Not testing custom transformers with cross_val_score to verify no leakage
